<a href="https://colab.research.google.com/github/mg15best/aerotwin-ai/blob/main/notebooks/01_aerotwin_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AeroTwin AI – Notebook 01

**Objetivo:** Construir un gemelo digital (digital twin) del ecosistema aeronáutico español utilizando datos abiertos de AENA y ENAIRE junto con técnicas de IA generativa (RAG).

**Fuentes de datos:**
- AENA – estadísticas de pasajeros (`data/raw/aena_passengers/`)
- AENA – directorio de aerolíneas (`data/raw/aena_airlines/`)
- AENA – metadatos de aeropuertos (`data/raw/aena_airports/`)
- ENAIRE – publicación de información aeronáutica AIP (`data/raw/enaire_aip/`)

## 0. Configuración del entorno

In [1]:
!pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [24]:
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Pega aquí tu Google Gemini API Key: ")

print("API key configurada correctamente:", bool(os.environ.get("GOOGLE_API_KEY")))

API key configurada correctamente: True


In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

response = llm.invoke("Responde solo con esta frase: Gemini funciona correctamente.")

print(response.content)

Gemini funciona correctamente.


In [27]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    output_dimensionality=768
)

test_vector = embeddings.embed_query("AeroTwin AI test embedding")

print("Embedding generado correctamente.")
print("Dimensión del vector:", len(test_vector))
print("Primeros 5 valores:", test_vector[:5])

Embedding generado correctamente.
Dimensión del vector: 768
Primeros 5 valores: [-0.037928145, 0.00288188, -0.017322036, -0.011273562, -0.06233103]


In [28]:
%cd /content/aerotwin-ai

!git pull

print("\nContenido actual de .env.example:")
with open(".env.example", "r", encoding="utf-8") as f:
    print(f.read())

/content/aerotwin-ai
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.08 KiB | 1.08 MiB/s, done.
From https://github.com/mg15best/aerotwin-ai
   d09c0cd..c1afcaa  main       -> origin/main
Updating d09c0cd..c1afcaa
Fast-forward
 ...ide_july_2026.pdf.pdf => aena_price_guide_july_2026.pdf} | Bin
 1 file changed, 0 insertions(+), 0 deletions(-)
 rename data/raw/aena_airlines/{aena_price_guide_july_2026.pdf.pdf => aena_price_guide_july_2026.pdf} (100%)

Contenido actual de .env.example:
# Google Gemini API Key
# Required to run AeroTwin AI with Gemini LLM and Gemini Embeddings
GOOGLE_API_KEY=your_google_gemini_api_key_here

# Optional model configuration
GEMINI_MODEL=gemini-2.5-flash
GEMINI_EMBEDDING_MODEL=gemini-embedding-2-preview
GEMINI_EMBEDDING_DIMENSIONS=768



In [29]:
from pathlib import Path

RAW_DATA_DIR = Path("data/raw")

document_files = []

for path in RAW_DATA_DIR.rglob("*"):
    if path.is_file():
        if "support_optional" in str(path):
            continue
        if path.name == ".gitkeep":
            continue
        if path.suffix.lower() in [".pdf", ".html", ".htm"]:
            document_files.append(path)

print("Documentos principales encontrados para el RAG:", len(document_files))

for file in document_files:
    print("-", file)

Documentos principales encontrados para el RAG: 18
- data/raw/aena_passengers/aena_tfs_information_meeting_points.html
- data/raw/aena_passengers/aena_passengers_general.html
- data/raw/aena_passengers/aena_tfs_special_needs.html
- data/raw/aena_passengers/aena_mad_passenger_homepage.html
- data/raw/aena_passengers/aena_passengers_prm_general.html
- data/raw/aena_passengers/aena_mad_airport_guide_maps.html
- data/raw/aena_airlines/aena_price_guide_july_2026.pdf
- data/raw/aena_airlines/aena_mad_business_profile.html
- data/raw/aena_airlines/aena_airlines_marketing_support.html
- data/raw/aena_airlines/aena_airlines_portal.html
- data/raw/aena_airlines/aena_airlines_new_routes.html
- data/raw/aena_airlines/aena_airlines_commercial_incentives.html
- data/raw/aena_airlines/aena_airports_destinations_network.html
- data/raw/enaire_aip/enaire_aip_gcxo_tenerife_norte.pdf
- data/raw/enaire_aip/enaire_aip_spain_general.html
- data/raw/enaire_aip/enaire_aip_lemd_madrid_barajas.html
- data/raw/e

In [30]:
%cd /content/aerotwin-ai
!git pull

/content/aerotwin-ai
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 8.50 KiB | 2.83 MiB/s, done.
From https://github.com/mg15best/aerotwin-ai
   c1afcaa..a8fb46f  main       -> origin/main
Updating c1afcaa..a8fb46f
Fast-forward
 notebooks/01_aerotwin_ai.ipynb | 941 ++++++++++++++++++++++++++++++-----------
 1 file changed, 686 insertions(+), 255 deletions(-)


## 1. Carga de datos

## 1. Carga y preparación de documentos

En este apartado se cargan los documentos oficiales que forman la base de conocimiento de AeroTwin AI.

El proyecto utiliza documentos en formato PDF y HTML procedentes de Aena y ENAIRE/AIP.  
Se excluye la carpeta `support_optional/` para mantener el RAG principal centrado en las fuentes documentales más relevantes.

Cada documento se carga con metadatos para identificar:

- Archivo de origen.
- Carpeta de origen.
- Tipo de documento.
- Público objetivo principal: pasajeros, industria turística/aerolíneas o información aeronáutica.

In [32]:
from pathlib import Path
from collections import Counter

from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

RAW_DATA_DIR = Path("data/raw")

EXCLUDED_FOLDERS = ["support_optional"]
SUPPORTED_EXTENSIONS = [".pdf", ".html", ".htm"]

document_files = []

for path in RAW_DATA_DIR.rglob("*"):
    if not path.is_file():
        continue

    if path.name == ".gitkeep":
        continue

    if any(excluded in path.parts for excluded in EXCLUDED_FOLDERS):
        continue

    if path.suffix.lower() in SUPPORTED_EXTENSIONS:
        document_files.append(path)

document_files = sorted(document_files)

print("Documentos principales encontrados para el RAG:", len(document_files))
print()

for file in document_files:
    print("-", file)

print("\nResumen por carpeta:")
folder_counter = Counter(str(file.parent) for file in document_files)

for folder, count in folder_counter.items():
    print(f"{folder}: {count} documentos")

Documentos principales encontrados para el RAG: 18

- data/raw/aena_airlines/aena_airlines_commercial_incentives.html
- data/raw/aena_airlines/aena_airlines_marketing_support.html
- data/raw/aena_airlines/aena_airlines_new_routes.html
- data/raw/aena_airlines/aena_airlines_portal.html
- data/raw/aena_airlines/aena_airports_destinations_network.html
- data/raw/aena_airlines/aena_mad_business_profile.html
- data/raw/aena_airlines/aena_price_guide_july_2026.pdf
- data/raw/aena_passengers/aena_mad_airport_guide_maps.html
- data/raw/aena_passengers/aena_mad_passenger_homepage.html
- data/raw/aena_passengers/aena_passengers_general.html
- data/raw/aena_passengers/aena_passengers_prm_general.html
- data/raw/aena_passengers/aena_tfs_information_meeting_points.html
- data/raw/aena_passengers/aena_tfs_special_needs.html
- data/raw/enaire_aip/enaire_aip_gcts_tenerife_sur.pdf
- data/raw/enaire_aip/enaire_aip_gcxo_tenerife_norte.pdf
- data/raw/enaire_aip/enaire_aip_lebl_barcelona.html
- data/raw/en

In [33]:
def infer_audience_from_path(path: Path) -> str:
    """
    Infers the main audience of a document from its folder.
    """
    path_parts = set(path.parts)

    if "aena_passengers" in path_parts:
        return "passenger"

    if "aena_airlines" in path_parts:
        return "travel_industry"

    if "enaire_aip" in path_parts:
        return "aviation_professional"

    return "general"


def load_html_document(path: Path) -> Document:
    """
    Loads and cleans a local HTML document.
    """
    html = path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines()]
    text = "\n".join(line for line in lines if line)

    return Document(
        page_content=text,
        metadata={
            "source": str(path),
            "source_file": path.name,
            "source_folder": path.parent.name,
            "file_type": "html",
            "audience": infer_audience_from_path(path),
        },
    )


def load_pdf_documents(path: Path) -> list[Document]:
    """
    Loads a local PDF document page by page.
    """
    loader = PyPDFLoader(str(path))
    pages = loader.load()

    for page_number, page in enumerate(pages, start=1):
        page.metadata.update(
            {
                "source": str(path),
                "source_file": path.name,
                "source_folder": path.parent.name,
                "file_type": "pdf",
                "audience": infer_audience_from_path(path),
                "page": page_number,
            }
        )

    return pages


loaded_documents = []

for file_path in document_files:
    if file_path.suffix.lower() == ".pdf":
        loaded_documents.extend(load_pdf_documents(file_path))

    elif file_path.suffix.lower() in [".html", ".htm"]:
        loaded_documents.append(load_html_document(file_path))

print("Documentos cargados en LangChain:", len(loaded_documents))

print("\nEjemplo de metadatos del primer documento:")
print(loaded_documents[0].metadata)

print("\nVista previa del contenido:")
print(loaded_documents[0].page_content[:1000])

Documentos cargados en LangChain: 160

Ejemplo de metadatos del primer documento:
{'source': 'data/raw/aena_airlines/aena_airlines_commercial_incentives.html', 'source_file': 'aena_airlines_commercial_incentives.html', 'source_folder': 'aena_airlines', 'file_type': 'html', 'audience': 'travel_industry'}

Vista previa del contenido:
Commercial Incentives | Summer and Winter Season | Aena
Incentives
Aena commercial incentives
breadcrumb-header
Discover the incentives and discounts Aena offers airlines
Incentive for airports with more than 3,5 million passengers
Apply before the end of the season that entitles the generation of the incentive
Incentive for airports with more than 3,5 million passengers (PDF - 339.71 kB)
Incentive on routes to Asia
Apply before the end of the season that entitles the generation of the incentive
Incentive on routes to Asia (PDF - 215.54 kB)
Incentive for airports with less than 3,5 million passengers
Apply before the end of the season that entitles the gener

## 2. División de documentos en chunks

Una vez cargados los documentos, se dividen en fragmentos más pequeños llamados chunks.

Este paso es necesario porque los modelos de embeddings y el sistema RAG funcionan mejor cuando los textos se dividen en unidades de información manejables.

Se utiliza `RecursiveCharacterTextSplitter`, manteniendo cierto solapamiento entre fragmentos para no perder contexto entre partes consecutivas del documento.

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(loaded_documents)

# Eliminar posibles chunks vacíos
chunks = [chunk for chunk in chunks if chunk.page_content.strip()]

# Añadir metadatos de control
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = index
    chunk.metadata["chunk_length"] = len(chunk.page_content)

print("Número total de chunks generados:", len(chunks))

print("\nChunks por audiencia:")
audience_counter = Counter(chunk.metadata.get("audience", "unknown") for chunk in chunks)

for audience, count in audience_counter.items():
    print(f"- {audience}: {count}")

print("\nChunks por carpeta:")
folder_counter = Counter(chunk.metadata.get("source_folder", "unknown") for chunk in chunks)

for folder, count in folder_counter.items():
    print(f"- {folder}: {count}")

print("\nEjemplo de chunk:")
print("Metadatos:", chunks[0].metadata)
print("\nContenido:")
print(chunks[0].page_content[:1000])

Número total de chunks generados: 969

Chunks por audiencia:
- travel_industry: 296
- passenger: 19
- aviation_professional: 654

Chunks por carpeta:
- aena_airlines: 296
- aena_passengers: 19
- enaire_aip: 654

Ejemplo de chunk:
Metadatos: {'source': 'data/raw/aena_airlines/aena_airlines_commercial_incentives.html', 'source_file': 'aena_airlines_commercial_incentives.html', 'source_folder': 'aena_airlines', 'file_type': 'html', 'audience': 'travel_industry', 'chunk_id': 0, 'chunk_length': 924}

Contenido:
Commercial Incentives | Summer and Winter Season | Aena
Incentives
Aena commercial incentives
breadcrumb-header
Discover the incentives and discounts Aena offers airlines
Incentive for airports with more than 3,5 million passengers
Apply before the end of the season that entitles the generation of the incentive
Incentive for airports with more than 3,5 million passengers (PDF - 339.71 kB)
Incentive on routes to Asia
Apply before the end of the season that entitles the generation of t

## 3. Creación de la base vectorial con ChromaDB

En este apartado se crea la base de datos vectorial del proyecto.

Cada chunk generado en el apartado anterior se transforma en un embedding mediante Gemini Embeddings. Después, esos vectores se almacenan en ChromaDB para permitir búsquedas semánticas.

La carpeta `chroma_db/` se genera automáticamente al ejecutar el notebook y no se sube al repositorio, ya que puede regenerarse a partir de los documentos originales.

In [2]:
import shutil
from pathlib import Path

from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

CHROMA_DIR = Path("chroma_db")
COLLECTION_NAME = "aerotwin_ai"

# Modelo de embeddings validado previamente en Colab
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    output_dimensionality=768
)

# Para evitar duplicados si se vuelve a ejecutar la celda, reconstruimos la base desde cero
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR),
    collection_name=COLLECTION_NAME
)

print("Base vectorial creada correctamente.")
print("Directorio de ChromaDB:", CHROMA_DIR)
print("Colección:", COLLECTION_NAME)
print("Número de vectores almacenados:", vectorstore._collection.count())

ModuleNotFoundError: No module named 'langchain_chroma'

## 4. Pipeline RAG – AeroTwin AI

Combinamos el contexto extraído del vector store con un LLM para responder preguntas sobre el ecosistema aeronáutico.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

SYSTEM_PROMPT = """
Eres AeroTwin AI, un asistente especializado en el ecosistema aeronáutico español.
Utiliza únicamente el contexto proporcionado para responder las preguntas.
Si no conoces la respuesta, di que no tienes información suficiente.

Contexto:
{context}

Pregunta: {question}
Respuesta:"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=SYSTEM_PROMPT,
)

# llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=0)

# Cargar índice guardado
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# vector_store = FAISS.load_local(
#     ROOT / "data" / "processed" / "faiss_aip_index",
#     embeddings,
#     allow_dangerous_deserialization=True,
# )

# qa_chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     chain_type="stuff",
#     retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
#     chain_type_kwargs={"prompt": prompt},
# )

print("Pipeline RAG – pendiente de índice vectorial.")

## 5. Preguntas de demostración

Ejecuta las preguntas del fichero `outputs/demo_questions.md` contra el pipeline RAG.

In [ ]:
demo_questions = [
    "¿Cuál fue el aeropuerto español con mayor tráfico de pasajeros en 2023?",
    "¿Cuáles son los procedimientos de llegada instrumental (STAR) vigentes en Madrid-Barajas?",
    "¿Qué aerolíneas operan más rutas en los aeropuertos de AENA?",
]

for question in demo_questions:
    print(f"\n❓ {question}")
    # answer = qa_chain.run(question)
    # print(f"💡 {answer}")
    print("   → (pipeline RAG pendiente de datos)")

## 6. Próximos pasos

1. Descargar los datasets de AENA desde el [portal de datos abiertos](https://estadisticas.aena.es/estadisticas/SASPortal/main.do) y colocarlos en `data/raw/aena_passengers/`, `data/raw/aena_airlines/` y `data/raw/aena_airports/`.
2. Descargar los documentos AIP de ENAIRE desde [su portal oficial](https://aip.enaire.es/) y colocarlos en `data/raw/enaire_aip/`.
3. Completar la celda de indexación (sección 3) y ejecutarla para construir el índice FAISS.
4. Probar el pipeline RAG con las preguntas de demostración (sección 5).
5. Evaluar la calidad de las respuestas y ajustar los parámetros de recuperación.